# Interface de Data Discovery: Jupyter + SQL
Podemos transformar o nosso JupyterLab numa autêntica *Worksheet* de Data Warehouse como o **AWS Athena**.

Para isso, usamos a extensão Mágica do Python para habilitar blocos de código nativos SQL.

> **Modelo ativo:** Insurance Regression v1 — previsão de custo de seguro de saúde (`charges`)


In [19]:
# Carrega a extensão Mágica do SQL
%load_ext sql

# Inicializa a conexão SQLAlchemy com o seu Trino (Redshift local)
%sql trino://jovyan@trino:8080/minio

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


--- 
## Suas Consultas Livres (*Ad-Hoc*)
A partir de agora, as células que começam com `%%sql` deixam de ser Python e são enviadas **diretamente ao Trino**. O resultado é renderizado em HTML idêntico ao de uma ferramenta gráfica de Client SQL:

In [20]:
%%sql
SHOW SCHEMAS

Running query in 'trino://jovyan@trino:8080/minio'

Schema
information_schema
raw
trusted


In [21]:
%%sql

SHOW COLUMNS FROM minio.trusted.insurance


Running query in 'trino://jovyan@trino:8080/minio'

Column,Type,Extra,Comment
age,bigint,,
sex,varchar,,
bmi,double,,
children,bigint,,
smoker,varchar,,
region,varchar,,
charges,double,,
ingestion_date,timestamp(6),,


In [22]:
%%sql

SELECT *
FROM minio.trusted.insurance
LIMIT 20


Running query in 'trino://jovyan@trino:8080/minio'

age,sex,bmi,children,smoker,region,charges,ingestion_date
30,male,22.99,2,yes,northwest,17361.7661,2026-04-18 19:53:36.446182
24,male,32.7,0,yes,southwest,34472.841,2026-04-18 19:53:36.446182
24,male,25.8,0,no,southwest,1972.95,2026-04-18 19:53:36.446182
38,female,27.6,0,no,southwest,5383.536,2026-04-18 19:53:36.446182
18,male,33.77,1,no,southeast,1725.5523,2026-04-18 19:53:36.446182
28,male,33.0,3,no,southeast,4449.462,2026-04-18 19:53:36.446182
59,male,25.46,0,no,northwest,12124.9924,2026-04-18 19:53:36.446182
19,female,24.605,1,no,northwest,2709.24395,2026-04-18 19:53:36.446182
19,female,27.9,0,yes,southwest,16884.924,2026-04-18 19:53:36.446182
47,male,28.215,3,yes,northwest,24915.22085,2026-04-18 19:53:36.446182


In [23]:
%%sql
-- Top 5 segurados mais caros — fumantes, ordenados por custo
-- Funciona com ou sem encoding aplicado (smoker_enc ou smoker)
SELECT *
FROM minio.trusted.insurance
WHERE TRY(smoker_enc) = 1 OR (TRY(smoker_enc) IS NULL AND smoker = 'yes')
ORDER BY charges DESC
LIMIT 5


Running query in 'trino://jovyan@trino:8080/minio'

RuntimeError: (trino.exceptions.TrinoUserError) TrinoUserError(type=USER_ERROR, name=COLUMN_NOT_FOUND, message="line 3:11: Column 'smoker_enc' cannot be resolved", query_id=20260418_184434_00135_83g9d)
[SQL: SELECT *
FROM minio.trusted.insurance
WHERE TRY(smoker_enc) = 1 OR (TRY(smoker_enc) IS NULL AND smoker = 'yes')
ORDER BY charges DESC
LIMIT 5]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [ ]:
%%sql
SELECT COUNT(*)
FROM minio.trusted.insurance
WHERE ingestion_date IS NULL


In [ ]:
%%sql
-- Distribuição de registros por camada (RAW vs TRUSTED)
SELECT 
    'RAW (Bruto/Sujo)' AS camada,
    COUNT(*) AS total_registros
FROM minio.raw.insurance

UNION ALL

SELECT 
    'TRUSTED (Curado/Limpo)' AS camada,
    COUNT(*) AS total_registros
FROM minio.trusted.insurance

ORDER BY camada


In [ ]:
%%sql
-- Custo médio por faixa etária (RAW)
SELECT 
    age,
    ROUND(AVG(charges), 2) AS avg_charges,
    COUNT(*) AS total
FROM minio.raw.insurance
GROUP BY age
ORDER BY age


In [ ]:
%%sql
-- Custo médio por fumante vs não-fumante (smoker: 'no'=não, 'yes'=sim)
SELECT 
    smoker,
    ROUND(AVG(charges), 2) AS avg_charges,
    COUNT(*) AS total
FROM minio.trusted.insurance
GROUP BY smoker
ORDER BY smoker


In [ ]:
#%%sql
#DELETE FROM minio.trusted.insurance


In [ ]:
#%%sql
# TRUNCATE TABLE minio.trusted.insurance
